# Control de potencia en el canal de interferencia de K usuarios
## WMMSE vs. SFP vs. una red neuronal MLP

**Objetivo del laboratorio.** Comparar tres enfoques para asignar potencia de transmisión en una red inalámbrica donde todos los usuarios interfieren entre sí:

1. **WMMSE** (Weighted Minimum Mean Square Error): algoritmo iterativo clásico.
2. **SFP** (Sequential Fractional Programming): programación fraccional con transformada cuadrática.
3. **MLP** (Perceptrón Multicapa): una red neuronal que *aprende* a imitar al algoritmo óptimo.

Al final podrás responder: ¿cuánta tasa se pierde al usar la red? ¿cuánto tiempo se gana?

---
## 1. Introducción y formulación matemática

### 1.1 Intuición del problema

Imagina **K parejas transmisor–receptor** hablando al mismo tiempo en la misma banda de frecuencia (como varios routers WiFi en un edificio). Cada transmisor $k$ quiere comunicarse con *su* receptor $k$, pero su señal también llega, atenuada, a los otros receptores y actúa como **interferencia**.

- Si todos transmiten a máxima potencia, la interferencia mutua es enorme y la tasa total cae.
- Si alguien baja su potencia, interfiere menos a los demás, pero también transmite menos información.

**El control de potencia** busca el balance óptimo: ¿cuánta potencia $p_k$ debe usar cada transmisor? Este problema es **no convexo** (NP-difícil en general), por eso se usan algoritmos iterativos que encuentran buenos óptimos locales.

### 1.2 Modelo del sistema

El receptor $k$ observa

$$
y_k = h_{kk}\,x_k + \sum_{j \neq k} h_{kj}\,x_j + n_k, \qquad n_k \sim \mathcal{CN}(0,\sigma^2),
$$

donde $h_{kj}$ es el coeficiente de canal del **transmisor $j$ al receptor $k$** (en nuestro caso, desvanecimiento Rayleigh: $h_{kj}\sim\mathcal{CN}(0,1)$), $x_k$ es el símbolo enviado y $p_k = \mathbb{E}\big[|x_k|^2\big]$ es la potencia del transmisor $k$, con la restricción $0 \le p_k \le P_{\max}$.

El **SINR** (Signal-to-Interference-plus-Noise Ratio) del usuario $k$ es

$$
\boxed{\;\text{SINR}_k(\mathbf{p}) = \frac{|h_{kk}|^2\,p_k}{\sum_{j\neq k}|h_{kj}|^2\,p_j + \sigma^2}\;}
$$

*Numerador*: potencia útil recibida. *Denominador*: interferencia de los demás + ruido térmico.

### 1.3 Función objetivo: maximizar la tasa global (Sum-Rate)

$$
\boxed{\;
\max_{\mathbf{p}} \;\; R(\mathbf{p}) = \sum_{k=1}^{K} \log_2\!\big(1+\text{SINR}_k(\mathbf{p})\big)
\quad \text{s.t.} \quad 0 \le p_k \le P_{\max},\;\; k=1,\dots,K
\;}
$$

### 1.4 Equivalencia MMSE (idea central de WMMSE)

Supongamos que el transmisor envía $x_k = v_k s_k$ con $s_k$ de varianza unitaria (así $p_k=v_k^2$) y el receptor estima el símbolo con una ganancia escalar $u_k$: $\hat{s}_k = u_k y_k$. El **error cuadrático medio** es

$$
e_k = \mathbb{E}\big[|\hat s_k - s_k|^2\big] = \big(1 - u_k h_{kk} v_k\big)^2 + \sum_{j\neq k} u_k^2 |h_{kj}|^2 v_j^2 + \sigma^2 u_k^2 .
$$

Si elegimos el receptor óptimo (MMSE) $u_k^\star = \dfrac{h_{kk} v_k}{\sum_j |h_{kj}|^2 v_j^2+\sigma^2}$, se obtiene

$$
e_k^\star = \frac{1}{1+\text{SINR}_k}
\quad\Longrightarrow\quad
\log_2(1+\text{SINR}_k) = -\log_2 e_k^\star .
$$

Por lo tanto, **maximizar la tasa equivale a minimizar $-\log e_k$**. Usando la identidad $\;-\log e = \max_{w>0}\,\big(-we + \log w + 1\big)$, el problema se convierte en uno **bloque a bloque convexo**:

$$
\min_{\{w_k,u_k,v_k\}} \;\sum_{k=1}^{K} \big(w_k\,e_k - \log w_k\big)
\quad \text{s.t.} \quad 0\le v_k\le \sqrt{P_{\max}} .
$$

Se optimiza cíclicamente sobre $\mathbf{u}$, luego $\mathbf{w}$ y luego $\mathbf{v}$ (cada paso tiene solución cerrada).

### 1.5 Transformación cuadrática en SFP

La tasa es una suma de logaritmos de **cocientes** (el SINR). La **programación fraccional (FP)** los reescribe en dos pasos:

**(i) Transformada dual de Lagrange** — introduce una variable auxiliar $\gamma_k$ por usuario:

$$
\sum_k \log(1+\gamma_k) - \sum_k \gamma_k + \sum_k \frac{(1+\gamma_k)\,|h_{kk}|^2 p_k}{\sum_{j}|h_{kj}|^2 p_j+\sigma^2}.
$$

**(ii) Transformada cuadrática** — para un cociente $\frac{A^2}{B}$ se usa $\;\frac{A^2}{B} = \max_y\,\big(2yA - y^2 B\big)$, lo que elimina las fracciones:

$$
f_q(\mathbf{p},\boldsymbol\gamma,\mathbf{y}) = \sum_k \log(1+\gamma_k) - \sum_k\gamma_k
+ \sum_k\Big(2y_k\sqrt{(1+\gamma_k)|h_{kk}|^2 p_k} - y_k^2\Big(\sum_j |h_{kj}|^2 p_j + \sigma^2\Big)\Big).
$$

Optimizando de forma **secuencial** cada variable (con las otras fijas) se obtienen actualizaciones cerradas:

$$
\gamma_k = \text{SINR}_k, \qquad
y_k = \frac{\sqrt{(1+\gamma_k)|h_{kk}|^2 p_k}}{\sum_j |h_{kj}|^2 p_j + \sigma^2}, \qquad
\sqrt{p_j} = \min\!\left\{\sqrt{P_{\max}},\; \frac{y_j\sqrt{(1+\gamma_j)}\,|h_{jj}|}{\sum_k y_k^2\,|h_{kj}|^2}\right\}.
$$

Cada iteración **no decrece** la tasa global (propiedad de *ascenso monótono*), convergiendo a un punto estacionario.

### 1.6 Enfoque de aprendizaje supervisado (MLP)

WMMSE y SFP requieren decenas o cientos de iteraciones con productos matriz-vector. Idea del *deep learning* (aprendizaje por imitación): **entrenar una red $f_\theta$ que aproxime el mapeo completo**

$$
f_\theta : \mathbf{H} \;\longmapsto\; \hat{\mathbf{p}} \approx \mathbf{p}^\star(\mathbf{H}),
$$

donde $\mathbf{p}^\star$ es la asignación producida por WMMSE (la "etiqueta" o *teacher*). Con un conjunto $\{(\mathbf{H}_i,\mathbf{p}^\star_i)\}_{i=1}^N$ se minimiza el error cuadrático medio:

$$
\min_\theta \;\frac{1}{N}\sum_{i=1}^{N}\big\|\,f_\theta(\mathbf{H}_i) - \mathbf{p}^\star_i\,\big\|_2^2 .
$$

La salida usa una sigmoide escalada, $\hat p_k = P_{\max}\,\sigma(z_k)$, para **garantizar** $0\le\hat p_k\le P_{\max}$. Una vez entrenada, la inferencia es **una sola pasada hacia adelante** (costo fijo y muy bajo).

> **Pregunta guía:** si el MLP solo imita a WMMSE, ¿puede superarlo en tasa? ¿Por qué o por qué no?

---
## 2. Configuración y simulación de canales

Definimos los parámetros del sistema y una función que genera **muchas realizaciones** de canal Rayleigh. Almacenamos la **magnitud** $|h_{kj}|$ en una matriz $\mathbf{H}\in\mathbb{R}^{K\times K}$ por muestra (fila $k$ = receptor, columna $j$ = transmisor). La diagonal contiene los canales *directos* y los elementos fuera de la diagonal los canales de *interferencia*.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "font.size": 11})

# ---------------- Parámetros del sistema ----------------
N_users     = 4      # K: número de pares transmisor-receptor
P_max       = 1.0    # potencia máxima por transmisor
noise_power = 0.1    # varianza del ruido sigma^2

print(f"K = {N_users} usuarios | P_max = {P_max} | sigma^2 = {noise_power}")

In [ ]:
def generate_channels(n_samples, n_users=N_users, rng=None):
    '''
    Genera canales con desvanecimiento Rayleigh.

    h_kj ~ CN(0,1)  ->  |h_kj| sigue una distribución Rayleigh.
    Devuelve un arreglo H de forma (n_samples, K, K) con las magnitudes |h_kj|
    (fila k = receptor, columna j = transmisor).
    '''
    rng = np.random.default_rng() if rng is None else rng
    real = rng.standard_normal((n_samples, n_users, n_users))
    imag = rng.standard_normal((n_samples, n_users, n_users))
    h = (real + 1j * imag) / np.sqrt(2)
    return np.abs(h)


def sum_rate(H, p, noise=noise_power):
    '''Tasa global (bits/s/Hz) para cada muestra del lote. H: (B,K,K), p: (B,K).'''
    G = H ** 2                                          # ganancias de potencia |h|^2
    total = np.einsum('bkj,bj->bk', G, p) + noise       # señal + interferencia + ruido
    signal = np.einsum('bkk->bk', G) * p                # señal útil
    sinr = signal / (total - signal)                    # SINR_k
    return np.log2(1.0 + sinr).sum(axis=1)


rng = np.random.default_rng(SEED)
N_train, N_test = 20000, 2000
H_train = generate_channels(N_train, rng=rng)
H_test  = generate_channels(N_test,  rng=rng)
print("H_train:", H_train.shape, "| H_test:", H_test.shape)

# Un vistazo a una realización
np.set_printoptions(precision=3, suppress=True)
print("\nEjemplo de matriz de canal |H| (diagonal = enlace directo):\n", H_test[0])

In [ ]:
# Visualización: los canales directos e interferentes tienen la misma estadística Rayleigh
fig, ax = plt.subplots()
diag = np.einsum('bkk->bk', H_train).ravel()
mask = ~np.eye(N_users, dtype=bool)
offd = H_train[:, mask].ravel()
ax.hist(diag, bins=60, density=True, alpha=0.6, label="directo $|h_{kk}|$")
ax.hist(offd, bins=60, density=True, alpha=0.6, label="interferencia $|h_{kj}|$")
x = np.linspace(0, 3.5, 200)
ax.plot(x, 2 * x * np.exp(-x**2), "k--", label="Rayleigh teórica")
ax.set_xlabel("magnitud del canal"); ax.set_ylabel("densidad"); ax.legend()
plt.show()

---
## 3. Algoritmo WMMSE

WMMSE actualiza **tres variables** de forma cíclica. Con $v_k=\sqrt{p_k}$ y $h_{kj}$ real positivo (magnitud):

**Paso 1 — Receptor MMSE $u_k$** (fijados $\mathbf{v}$):
$$
u_k = \frac{h_{kk}\,v_k}{\sum_{j=1}^{K} h_{kj}^2\,v_j^2 + \sigma^2}
$$
Es el filtro que minimiza el error de estimación del usuario $k$ dada la potencia actual de todos.

**Paso 2 — Peso $w_k$** (fijados $\mathbf{u},\mathbf{v}$):
$$
w_k = \frac{1}{1 - u_k\,h_{kk}\,v_k} \;=\; \frac{1}{e_k}
$$
Es el inverso del MSE: los usuarios con **mayor error** reciben **más peso** en el siguiente paso.

**Paso 3 — Transmisor $v_k$** (fijados $\mathbf{u},\mathbf{w}$):
$$
v_k = \min\!\left\{\sqrt{P_{\max}},\;\frac{w_k\,u_k\,h_{kk}}{\sum_{j=1}^{K} w_j\,u_j^2\,h_{jk}^2}\right\}
$$
El denominador mide **cuánta interferencia causa el transmisor $k$ a todos los receptores** (ponderada por su importancia). El `min` impone la restricción de potencia.

Se repite hasta que $\mathbf{v}$ deja de cambiar. Cada iteración no disminuye la tasa global.

In [ ]:
def wmmse_power(H, P_max=P_max, noise=noise_power, n_iter=100, tol=1e-5, return_history=False):
    '''
    WMMSE vectorizado para un lote de canales.
    H: (B,K,K) con |h_kj| (fila = receptor, columna = transmisor).
    Devuelve p: (B,K) y, opcionalmente, el historial de la tasa media por iteración.
    '''
    B, K, _ = H.shape
    G = H ** 2                                  # |h_kj|^2
    hd = np.einsum('bkk->bk', H)                # canales directos h_kk
    v_max = np.sqrt(P_max)
    v = np.full((B, K), v_max)                  # inicialización: potencia máxima
    history = []

    for _ in range(n_iter):
        v_old = v
        # Paso 1: receptor MMSE
        u = hd * v / (np.einsum('bkj,bj->bk', G, v**2) + noise)
        # Paso 2: peso (inverso del MSE)
        w = 1.0 / (1.0 - u * hd * v)
        # Paso 3: transmisor (con proyección a [0, sqrt(P_max)])
        num = w * u * hd
        den = np.einsum('bjk,bj->bk', G, w * u**2)     # sum_j w_j u_j^2 h_jk^2
        v = np.minimum(v_max, num / den)

        if return_history:
            history.append(sum_rate(H, v**2, noise).mean())
        if np.max(np.abs(v - v_old)) < tol:
            break

    p = v ** 2
    return (p, np.array(history)) if return_history else p

In [ ]:
# Prueba rápida de WMMSE: una realización de canal y evolución de la tasa
H_one = H_test[:1]
p_w, hist_w = wmmse_power(H_one, n_iter=30, tol=0, return_history=True)

print("Potencias WMMSE :", p_w[0])
print("Tasa WMMSE      :", sum_rate(H_one, p_w)[0].round(3), "bits/s/Hz")
print("Tasa full-power :", sum_rate(H_one, np.full((1, N_users), P_max))[0].round(3), "bits/s/Hz")

plt.plot(np.arange(1, len(hist_w) + 1), hist_w, "o-")
plt.xlabel("iteración"); plt.ylabel("Sum-rate (bits/s/Hz)")
plt.title("Convergencia monótona de WMMSE (una realización)")
plt.show()

---
## 4. Algoritmo SFP (Sequential Fractional Programming)

Implementamos las tres actualizaciones secuenciales de la sección 1.5:

1. **$\gamma_k$** = SINR actual.
2. **$y_k$** (transformada cuadrática): $\;y_k=\dfrac{\sqrt{(1+\gamma_k)\,G_{kk}\,p_k}}{\sum_j G_{kj}p_j+\sigma^2}$, con $G_{kj}=|h_{kj}|^2$.
3. **$p_k$**: $\;\sqrt{p_j}=\min\Big\{\sqrt{P_{\max}},\ \dfrac{y_j\sqrt{(1+\gamma_j)G_{jj}}}{\sum_k y_k^2\,G_{kj}}\Big\}$.

> **Observación para discutir:** compara esta actualización con el paso 3 de WMMSE. Ambas dividen "valor de la señal propia" entre "interferencia causada a los demás". Se ha demostrado que, con la elección adecuada de variables, WMMSE y FP son **algoritmos equivalentes en esencia**, pero la formulación FP es más general (admite otras funciones objetivo y otros canales).

In [ ]:
def sfp_power(H, P_max=P_max, noise=noise_power, n_iter=100, tol=1e-5, return_history=False):
    '''
    Programación fraccional secuencial (transformada dual de Lagrange + cuadrática).
    H: (B,K,K) con |h_kj|. Devuelve p: (B,K) y, opcionalmente, historial de tasa media.
    '''
    B, K, _ = H.shape
    G = H ** 2
    gd = np.einsum('bkk->bk', G)                # G_kk
    p = np.full((B, K), P_max)
    x_max = np.sqrt(P_max)
    history = []

    for _ in range(n_iter):
        p_old = p
        total = np.einsum('bkj,bj->bk', G, p) + noise            # denominador del SINR
        # Paso 1: gamma_k = SINR_k
        gamma = gd * p / (total - gd * p)
        # Paso 2: transformada cuadrática
        y = np.sqrt((1.0 + gamma) * gd * p) / total
        # Paso 3: actualización de la potencia
        num = y * np.sqrt((1.0 + gamma) * gd)
        den = np.einsum('bkj,bk->bj', G, y**2)                   # sum_k y_k^2 G_kj
        x = np.minimum(x_max, num / den)
        p = x ** 2

        if return_history:
            history.append(sum_rate(H, p, noise).mean())
        if np.max(np.abs(p - p_old)) < tol:
            break

    return (p, np.array(history)) if return_history else p

In [ ]:
# Comparación de convergencia WMMSE vs SFP sobre 200 canales (tasa media por iteración)
H_sub = H_test[:200]
_, hw = wmmse_power(H_sub, n_iter=40, tol=0, return_history=True)
_, hs = sfp_power(H_sub,   n_iter=40, tol=0, return_history=True)

plt.plot(np.arange(1, 41), hw, "o-", label="WMMSE")
plt.plot(np.arange(1, 41), hs, "s--", label="SFP")
plt.xlabel("iteración"); plt.ylabel("Sum-rate medio (bits/s/Hz)")
plt.title("Convergencia: WMMSE vs SFP"); plt.legend(); plt.show()

---
## 5. Red neuronal MLP aproximadora

### 5.1 Generación de etiquetas
Usamos **WMMSE como profesor** (*teacher*): calculamos $\mathbf{p}^\star_i$ para cada canal de entrenamiento. Este es el paso costoso, pero se hace **una sola vez, fuera de línea**.

### 5.2 Arquitectura
$$
\underbrace{K^2}_{\text{entrada: }\mathbf{H}} \to 128 \to 128 \to 64 \to \underbrace{K}_{\text{salida: }\hat{\mathbf p}},\qquad
\text{activaciones ReLU, salida } P_{\max}\,\sigma(\cdot).
$$
La matriz $\mathbf{H}$ se aplana y se estandariza (media 0, desviación 1) para facilitar el entrenamiento.

In [ ]:
# 5.1 Etiquetas óptimas con WMMSE (fuera de línea)
t0 = time.time()
P_train = wmmse_power(H_train, n_iter=100)
P_test_wmmse = wmmse_power(H_test, n_iter=100)
print(f"Etiquetas generadas en {time.time() - t0:.2f} s")

# Normalización de entradas (con estadísticas SOLO del conjunto de entrenamiento)
X_train_raw = H_train.reshape(N_train, -1)
mu, sd = X_train_raw.mean(axis=0), X_train_raw.std(axis=0)
prep = lambda H: torch.tensor(((H.reshape(len(H), -1) - mu) / sd), dtype=torch.float32)

X_train, Y_train = prep(H_train), torch.tensor(P_train, dtype=torch.float32)
X_test = prep(H_test)

# Reservamos 10% del entrenamiento para validación
n_val = N_train // 10
train_ds = TensorDataset(X_train[n_val:], Y_train[n_val:])
val_ds   = TensorDataset(X_train[:n_val], Y_train[:n_val])
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=1024)

In [ ]:
class PowerMLP(nn.Module):
    def __init__(self, n_users=N_users, p_max=P_max, hidden=(128, 128, 64)):
        super().__init__()
        layers, d_in = [], n_users * n_users
        for h in hidden:
            layers += [nn.Linear(d_in, h), nn.ReLU()]
            d_in = h
        layers += [nn.Linear(d_in, n_users)]
        self.net = nn.Sequential(*layers)
        self.p_max = p_max

    def forward(self, x):
        return self.p_max * torch.sigmoid(self.net(x))     # garantiza 0 <= p <= P_max


device = "cuda" if torch.cuda.is_available() else "cpu"
model = PowerMLP().to(device)
print(model)
print("Parámetros entrenables:", sum(p.numel() for p in model.parameters()))

In [ ]:
# 5.2 Entrenamiento supervisado (MSE contra las potencias de WMMSE)
n_epochs = 60
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.3)
loss_fn = nn.MSELoss()
train_hist, val_hist = [], []

for epoch in range(n_epochs):
    model.train(); running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward(); optimizer.step()
        running += loss.item() * len(xb)
    scheduler.step()
    train_hist.append(running / len(train_ds))

    model.eval()
    with torch.no_grad():
        val_loss = sum(loss_fn(model(xb.to(device)), yb.to(device)).item() * len(xb)
                       for xb, yb in val_dl) / len(val_ds)
    val_hist.append(val_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Época {epoch+1:3d} | MSE entrenamiento {train_hist[-1]:.5f} | MSE validación {val_hist[-1]:.5f}")

plt.semilogy(train_hist, label="entrenamiento"); plt.semilogy(val_hist, label="validación")
plt.xlabel("época"); plt.ylabel("MSE"); plt.title("Curvas de aprendizaje"); plt.legend(); plt.show()

---
## 6. Comparación de resultados

Evaluamos sobre el **conjunto de prueba** (canales nunca vistos) con:

- **Sum-rate medio** de cada método,
- **Tiempo de cómputo** por muestra,
- **Distribución** de la razón $R_{\text{MLP}}/R_{\text{WMMSE}}$,
- *Líneas base*: potencia máxima para todos (*full power*) y potencia aleatoria.

> **Aviso de justicia en la medición de tiempos.** WMMSE y SFP aquí están vectorizados con NumPy (CPU) y el MLP con PyTorch; los tiempos dependen del hardware y de la implementación. Lo importante es el **orden de magnitud** y la diferencia en **tipo de costo**: iterativo (decenas de iteraciones) vs. una pasada directa.

In [ ]:
def timed(fn, *args, repeats=5, **kw):
    best, out = np.inf, None
    for _ in range(repeats):
        t0 = time.perf_counter(); out = fn(*args, **kw); best = min(best, time.perf_counter() - t0)
    return out, best

# --- Inferencia / cómputo sobre el conjunto de prueba ---
p_wmmse, t_wmmse = timed(wmmse_power, H_test, n_iter=100)
p_sfp,   t_sfp   = timed(sfp_power,   H_test, n_iter=100)

model.eval()
def mlp_predict(X):
    with torch.no_grad():
        return model(X.to(device)).cpu().numpy()
p_mlp, t_mlp = timed(mlp_predict, X_test)

# Líneas base
p_full = np.full((N_test, N_users), P_max)
p_rand = np.random.default_rng(0).uniform(0, P_max, (N_test, N_users))

R = {
    "WMMSE": sum_rate(H_test, p_wmmse),
    "SFP": sum_rate(H_test, p_sfp),
    "MLP": sum_rate(H_test, p_mlp),
    "Full power": sum_rate(H_test, p_full),
    "Aleatorio": sum_rate(H_test, p_rand),
}
times = {"WMMSE": t_wmmse, "SFP": t_sfp, "MLP": t_mlp}

print(f"{'Método':<12}{'Sum-rate medio':>16}{'% de WMMSE':>13}{'Tiempo/muestra (µs)':>22}")
for name, r in R.items():
    tt = f"{times[name] / N_test * 1e6:.1f}" if name in times else "—"
    print(f"{name:<12}{r.mean():>16.3f}{100 * r.mean() / R['WMMSE'].mean():>12.1f}%{tt:>22}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Sum-rate medio
names = list(R.keys()); means = [R[n].mean() for n in names]
axes[0].bar(names, means, color=["C0", "C1", "C2", "C7", "C7"])
axes[0].set_ylabel("Sum-rate medio (bits/s/Hz)"); axes[0].set_title("(a) Tasa global media")
axes[0].tick_params(axis="x", rotation=25)

# (b) Dispersión MLP vs WMMSE por muestra
axes[1].scatter(R["WMMSE"], R["MLP"], s=4, alpha=0.4)
lim = [0, max(R["WMMSE"].max(), R["MLP"].max()) * 1.05]
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set_xlabel("Sum-rate WMMSE"); axes[1].set_ylabel("Sum-rate MLP"); axes[1].set_title("(b) Muestra a muestra")

# (c) Histograma de la razón
ratio = R["MLP"] / R["WMMSE"]
axes[2].hist(ratio, bins=60, color="C2", alpha=0.8)
axes[2].axvline(ratio.mean(), color="k", ls="--", label=f"media = {ratio.mean():.3f}")
axes[2].set_xlabel("$R_{MLP} / R_{WMMSE}$"); axes[2].set_title("(c) Razón de tasas"); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Potencias para unas cuantas realizaciones: ¿qué estrategia aprendió la red?
idx = [0, 1, 2, 3]
fig, axes = plt.subplots(1, len(idx), figsize=(15, 3.3), sharey=True)
x = np.arange(N_users); w = 0.27
for ax, i in zip(axes, idx):
    ax.bar(x - w, p_wmmse[i], w, label="WMMSE")
    ax.bar(x,     p_sfp[i],   w, label="SFP")
    ax.bar(x + w, p_mlp[i],   w, label="MLP")
    ax.set_xticks(x); ax.set_xticklabels([f"Tx{k+1}" for k in x]); ax.set_title(f"Canal #{i}")
axes[0].set_ylabel("potencia $p_k$"); axes[0].legend()
plt.tight_layout(); plt.show()

### 6.1 Exploración interactiva (opcional)

Mueve los controles para ver cómo cambian las potencias y la tasa para **una** realización de canal. Observa que WMMSE y SFP suelen "apagar" a los usuarios con peor canal directo o que más interfieren (soluciones tipo *todo o nada*), y que el MLP intenta reproducir ese patrón.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def explore(sample=0, n_iter=10):
        H1 = H_test[sample:sample + 1]
        pw = wmmse_power(H1, n_iter=n_iter, tol=0)
        ps = sfp_power(H1, n_iter=n_iter, tol=0)
        pm = mlp_predict(X_test[sample:sample + 1])
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
        x = np.arange(N_users)
        ax[0].bar(x - 0.27, pw[0], 0.27, label="WMMSE"); ax[0].bar(x, ps[0], 0.27, label="SFP")
        ax[0].bar(x + 0.27, pm[0], 0.27, label="MLP"); ax[0].set_ylim(0, P_max * 1.05)
        ax[0].set_xticks(x); ax[0].set_xticklabels([f"Tx{k+1}" for k in x]); ax[0].legend()
        ax[0].set_title(f"Potencias (WMMSE/SFP con {n_iter} iteraciones)")
        vals = [sum_rate(H1, pw)[0], sum_rate(H1, ps)[0], sum_rate(H1, pm)[0]]
        ax[1].bar(["WMMSE", "SFP", "MLP"], vals, color=["C0", "C1", "C2"])
        ax[1].set_ylabel("Sum-rate (bits/s/Hz)"); ax[1].set_title("Tasa global")
        plt.tight_layout(); plt.show()

    widgets.interact(explore,
                     sample=widgets.IntSlider(0, 0, N_test - 1, 1, description="canal #"),
                     n_iter=widgets.IntSlider(10, 1, 100, 1, description="iteraciones"))
except ImportError:
    print("ipywidgets no está instalado: pip install ipywidgets (celda opcional).")

---
## 7. Conclusiones y preguntas de discusión

**Qué deberías observar**

- **WMMSE y SFP** alcanzan prácticamente la misma tasa (ambos llegan a puntos estacionarios similares), con convergencia monótona.
- El **MLP** logra típicamente una fracción alta de la tasa de WMMSE (orden del 90–99 % según el entrenamiento) con un costo de inferencia **mucho menor y constante**.
- Las líneas base (*full power* y aleatorio) quedan claramente por debajo: la interferencia **sí** importa.

**Preguntas para reflexionar**

1. ¿Por qué el MLP rara vez supera a su profesor WMMSE? ¿Qué cambiaría si entrenáramos **sin etiquetas** maximizando directamente $-R(f_\theta(\mathbf{H}))$ (aprendizaje no supervisado)?
2. ¿Cómo cambia la brecha MLP–WMMSE si aumentas `N_users`? ¿Y si reduces el tamaño del conjunto de entrenamiento a 2 000 muestras?
3. Cambia `noise_power` a 1.0 y a 0.01. ¿En qué régimen (limitado por ruido vs. limitado por interferencia) *full power* se acerca más al óptimo?
4. Si los canales cambian muy rápido (tiempo de coherencia corto), ¿qué ventaja práctica tiene el MLP?
5. Un MLP entrenado con $K=4$ no funciona con $K=5$. ¿Qué arquitecturas (p. ej. redes de grafos, GNN) resolverían este problema de generalización?

**Referencias**

- Shi, Razaviyayn, Luo, He, *"An Iteratively Weighted MMSE Approach to Distributed Sum-Utility Maximization for a MIMO Interfering Broadcast Channel"*, IEEE TSP, 2011.
- Shen, Yu, *"Fractional Programming for Communication Systems—Part I: Power Control and Beamforming"*, IEEE TSP, 2018.
- Sun, Chen, Shi, Hong, Gao, *"Learning to Optimize: Training Deep Neural Networks for Interference Management"*, IEEE TSP, 2018.